### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    1. Tracking agent behaviour with logging , analytics and debugging.

    2. Transforming prompts, tool selection and output formatting.

    3. Adding retries, fallbacks and early termination logic.

    4. Applying rate limits, guardraills and PII detection.

   ### Types of Middlewares
 1 Summarization

2. Human in loop 

3. Model call limits 

and many more


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("groqAPIKey")

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

#### MessageBased Summarization
agent = create_agent(
    model="groq:llama-3.1-8b-instant",   # 👈 IMPORTANT (provider prefix)
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger = ("messages",10),
            keep=("messages",4)
        )
    ],
    system_prompt="You are a helpful assistant. Call the tool ONLY once.After receiving result, respond to user directly.DO NOT call tool again."
)

In [4]:
#### Run with a thread ID
config= {"configurable": {"thread_id":"test-1"}}

# Alternative test data

questions = [
    "What is 2+2",
    "What is 4*2",
    "What is 5-2",
    "What is 1+2",
    "What is 2+2",
    "What is 3%2",
    "What is 2+2",
    "What is 4*2",
    "What is 5-2",
    "What is 1+2",
    "What is 2+2",
    "What is 3%2",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messagess: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='593b12d5-18ed-4b61-b32c-f2105b7f8981'), AIMessage(content='The result of 2+2 is 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 66, 'total_tokens': 78, 'completion_time': 0.020467781, 'completion_tokens_details': None, 'prompt_time': 0.003703875, 'prompt_tokens_details': None, 'queue_time': 0.047034355, 'total_time': 0.024171656}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d21bd-0522-7133-b1b8-dad070ba752d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 12, 'total_tokens': 78})]}
Messagess: 2
Messages: {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='593b12d5-18ed-4b61-b32c-f2105b7f898

### Summarization based on TOKEN SIZE

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_Hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1 .Grand Hotel - 5 star, $350$/night, spa, pool, gym
    2. Taja Hotel - 3 star, $50$/night pool, gym
    3. baja Hotel - 2 star, $40$/night spa, gym
    """


agent = create_agent(
    model = "groq:llama-3.1-8b-instant",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger = ("tokens",1000),
            keep=("tokens",200)
        ),
    ]
)

# Token Counter (approximate)
def count_Tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4   # 4 chars = 1 token

In [6]:
# Test Run

cities = ["Paris","London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_Tokens(response["messages"])
    print(f"{city}: ~ {tokens}, {len(response["messages"])}) messages")
    print(f"{(response['messages'])}")

Paris: ~ 565, 2) messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='f1efff0d-ecc5-4d71-93f9-7871243387a4'), AIMessage(content="I can provide you with some information about hotels in Paris. Please note that I'll be listing a few popular options, but there are many more to choose from depending on your budget and preferences.\n\n**Luxury Hotels:**\n\n1. **The Ritz Paris**: A 5-star hotel located on the Place Vendôme, known for its opulent decor and exceptional service.\n2. **Shangri-La Hotel, Paris**: A 5-star hotel located in the 16th arrondissement, offering stunning views of the Eiffel Tower and luxurious amenities.\n3. **Four Seasons Hotel George V Paris**: A 5-star hotel located in the 8th arrondissement, known for its elegant decor and exceptional service.\n\n**Mid-range Hotels:**\n\n1. **Hotel Plaza Athenee**: A 4-star hotel located in the 8th arrondissement, offering comfortable rooms and a beautiful garden.\n2. **Hotel Le Walt

KeyboardInterrupt: 

### Fraction
 Based on the context of the model
 

### InMemorySaver()

InMemorySaver() is a temporary checkpoint storage used in LangGraph/LangChain to:

✅ Save the state of your workflow/agent
✅ Allow step-by-step memory sharing
✅ Help resume execution from last step

⚠️ It stores data only in memory (RAM) → lost when the app restarts.

👉 In short:
It lets your AI “remember” progress during execution, but not permanently.

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_Hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1 .Grand Hotel - 5 star, $350$/night, spa, pool, gym
    2. Taja Hotel - 3 star, $50$/night pool, gym
    3. baja Hotel - 2 star, $40$/night spa, gym
    """


agent = create_agent(
    model = "groq:llama-3.1-8b-instant",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger = ("fraction",0.005),
            keep=("fraction",0.002)
        ),
    ]
)

# Token Counter (approximate)
def count_Tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4   # 4 chars = 1 token

# Test Run

cities = ["Paris","London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_Tokens(response["messages"])
    print(f"{city}: ~ {tokens}, {len(response["messages"])}) messages")
    print(f"{(response['messages'])}")

Paris: ~ 742, 2) messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b62fbefb-ae5a-4190-9c12-a0fa2312c29c'), AIMessage(content="Paris, the City of Light, is a popular destination for travelers from around the world. Here are some top-rated hotels in Paris, categorized by price range:\n\n**Luxury Hotels (€€€)**\n\n1. **Shangri-La Hotel, Paris** - A 5-star hotel located in the heart of Paris, with stunning views of the Eiffel Tower.\n2. **Four Seasons Hotel George V Paris** - A luxurious hotel with elegant rooms and exceptional service.\n3. **The Ritz Paris** - A classic hotel with luxurious rooms and a rich history.\n4. **Peninsula Paris** - A 5-star hotel with elegant rooms and a rooftop pool.\n5. **Plaza Athénée** - A luxurious hotel with elegant rooms and a Michelin-starred restaurant.\n\n**Boutique Hotels (€€)**\n\n1. **Le Pigalle** - A stylish hotel with boutique rooms and a lively atmosphere.\n2. **Hôtel Particulier Montmartre** -

### Human in the Loop Middleware

Pause agent execution for human approval, editing or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations require human approval (e.g database writes, financial transactions)
- Compliance workflows where human oversight is mandatory.
- Long-running conversation where human feedback guide the agent.

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """ Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"
@tool
def send_email_tool(recipient: str, subject: str, body: str)-> str:
    """ Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"



In [29]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decision":["approve","edit","reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [30]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content = "Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4fb9e50a-8c75-4d84-9898-d7cc21607064'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '06pjaxkhx', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.032630016, 'completion_tokens_details': None, 'prompt_time': 0.022868667, 'prompt_tokens_details': None, 'queue_time': 0.046055453, 'total_time': 0.055498683}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2698-0a50-7c31-bf76-48543d5befab-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body

In [32]:
from langgraph.types import Command
# Step 2 : Approve

if "__interrupt__" in result:
    print(" Paused ! Approving....")
    
    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type":"approve"}
                ]
            }
        ),
        config = config

    )

    print(f" Result: {result['messages'][-1].content}")


In [ ]:
### Example from claude
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq

# Tools
@tool
def read_email_tool(email_id: str) -> str:
    """Read an email by its ID"""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email"""
    print(f"\n📧 Sending email to: {recipient}")
    print(f"📧 Subject: {subject}")
    print(f"📧 Body: {body}")
    return f"Email sent to {recipient}"

# Initialize
llm = ChatGroq(model="llama-3.1-8b-instant")
memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=memory,
    interrupt_before=["tools"]
)

def run_with_approval(user_input: str, thread_id: str = "default"):
    """Run agent with human approval"""
    config = {"configurable": {"thread_id": thread_id}}
    
    # Step 1: Initial request
    print("\n" + "=" * 60)
    print("🤖 Processing your request...")
    print("=" * 60)
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    
    # Step 2: Check for interruption
    state = agent.get_state(config)
    
    if state.next:
        print("\n⏸️  Agent paused for approval")
        
        # Show tool details
        last_msg = result['messages'][-1]
        if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
            for tool_call in last_msg.tool_calls:
                print(f"\n🔧 Tool: {tool_call['name']}")
                print(f"🔧 Arguments:")
                for key, value in tool_call['args'].items():
                    print(f"   - {key}: {value}")
        
        # Get approval
        approval = input("\n❓ Approve this action? (yes/no): ").lower()
        
        if approval == "yes":
            print("\n✅ Approved! Executing...")
            result = agent.invoke(None, config=config)
            print(f"\n✅ Result: {result['messages'][-1].content}")
            return result
        else:
            print("\n❌ Rejected! Action cancelled.")
            return None
    else:
        print("\n✅ Completed without interruption")
        print(f"Result: {result['messages'][-1].content}")
        return result


# Usage
run_with_approval(
    "Send email to john@test.com with subject 'Meeting' and body 'See you tomorrow'"
)

C:\Users\pushk\AppData\Local\Temp\ipykernel_10608\4086449368.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



🤖 Processing your request...

⏸️  Agent paused for approval

🔧 Tool: send_email_tool
🔧 Arguments:
   - body: See you tomorrow
   - recipient: john@test.com
   - subject: Meeting

✅ Approved! Executing...

📧 Sending email to: john@test.com
📧 Subject: Meeting
📧 Body: See you tomorrow


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=read_email_tool>{"email_id": "ID of the email sent to john@test.com"}'}}

: 